In [3]:
from Z import *
from Poly import *
import numpy as np

In [52]:
''' Some number theoretic helper functions'''
def factor(n : int) -> list[list[int]]:
    assert n >= 1, 'factoring is defined for positive integers'
    result = []
    num = n
    with open('small_primes.txt') as file:
        for prime in file:
            p = int(prime[:-1])
            if num % p == 0:
                div = [p, 1]
                num = num // p
                while num % p == 0:
                    num //= p
                    div[1] += 1
                result.append(div)
            if num == 1: 
                return result
        raise ValueError('prime factors are too big to factor by trial division')
    
def mobius(n : int) -> int:
    assert n >= 1, 'mobius function is defined for positive integers'
    factors = factor(n)
    if any([exp > 1 for prime, exp in factors]):
        return 0
    else:
        return (-1) ** (len(factors) % 2)

def phi(n : int) -> int:
    assert n >= 1, 'totient function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= (prime - 1) * prime ** (exp - 1)
    return result

def rad(n : int) -> int:
    assert n >= 1, 'radical function is defined for positive integers'
    factors = factor(n)
    result = 1
    for prime, exp in factors:
        result *= prime
    return result

def divs(n : int) -> list[int]:
    assert n >= 1, 'divisors are defined for positive integers'
    if n == 1: return [1]
    factors = factor(n)
    p, e = factors[0]
    divisors = [p ** k for k in range(e + 1)]
    for prime, exp in factors[1:]:
        more_divs = [prime ** k * div for div in divisors 
                     for k in range(1, exp + 1)]
        divisors += more_divs
    return sorted(divisors)

def val(n : int, p : int) -> int:
    ''' max {k | p^k divides n}'''
    assert n >= 1, 'valuation is defined for positive integers'
    v, m = 0, n
    while m % p == 0:
        m //= p
        v += 1
    return v
    

In [39]:
''' Some specific polynomial operations as arrays'''
def compose(poly : np.ndarray, exp : int, neg : bool = False) -> np.ndarray:
    ''' poly(x) -> poly(x ^ exp)'''
    d = len(poly) - 1
    result = np.zeros(exp * d + 1, dtype=poly.dtype) 
    result[::exp] = poly
    if neg:
        result[exp::2 * exp] *= -1
    return result

def unity_mult_mod(poly: np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(d - 1, n - 1, -1):
        result[i] -= result[i - n]
    return result

def unity_div_mod(poly : np.ndarray, n : int, d : int) -> np.ndarray:
    ''' poly(x) -> poly(x)/(1 - x^n) modulo x^d'''
    # make the right length first
    result = poly[:d] if len(poly) >= d else np.pad(poly, (0, d - len(poly)))
    for i in range(n, d):
        result[i] += result[i - n]
    return result

In [53]:
def cyclo_odd_sq_free_composite(n : int, check=True) -> np.ndarray:
    ''' Phi_n(x) given that n = rad(n) is odd and composite'''
    if check:
        # check that n is of correct form
        factors = factor(n)
        assert (len(factors) > 1 
                and factors[0][0] > 2 
                and all([exp == 1 for p, exp in factors])
                ), 'odd sq free composite means product of distinct odd primes'
    
    D = phi(n) // 2 # is necessarily odd
    result = np.zeros(D + 1, dtype=object) # first half of coeff since symmetric
    result[0] = 1
    for div in divs(n)[:-1]:
        if mobius(n // div) == 1:
            unity_mult_mod(result, div, D + 1) 
        else:
            unity_div_mod(result, div, D + 1)
            
    reverse = result[-2::-1] # first phi(n)/2 - 1 coeff in reverse order
    combined = np.concatenate((result, reverse), axis=0) # palindromic
    return combined

def cyclo(n : int) -> np.ndarray:
    if n == 1: # trivial case
        return np.array([-1, 1], dtype=object)
    N = rad(n)
    if N == 2: # n = 2^k ==> phi_n(x) = x^{2^{k-1}} + 1
        result = [int(k in {0, n//2}) for k in range(1 + n // 2)] 
        return np.array(result, dtype=object)
    elif N % 2 == 0: # n = 2^k * (odd) ==> phi_n(x)=phi_{odd}(-x^{2^{k-1}})
        v = val(n, 2)
        return compose(cyclo(n // 2**v), 2**(v - 1), neg=True)
    elif is_prime(N):
        phi_p = np.array([1 for _ in range(N)], dtype=object)
        v = val(n, N)
        return compose(phi_p, N ** (v - 1))
    else: # phi_n(x) = phi_{rad(n)}(x^{n/rad(n)})
        return compose(cyclo_odd_sq_free_composite(N), n // N)

In [58]:
cyclo(105)

array([1, 1, 1, 0, 0, -1, -1, -2, -1, -1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0,
       -1, 0, -1, 0, -1, 0, -1, 0, -1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, -1,
       -1, -2, -1, -1, 0, 0, 1, 1, 1], dtype=object)